# Notebook 3: HerWell Chatbot (Integration)

**Dependencies:** Execute `01_config.ipynb` and `02_rag_system.ipynb` first so that configuration variables and the `RAGSystem` class are available in the kernel session.

**Responsibilities:**
1. Invoke the triage model to classify the urgency of the user's query.
2. Route the query based on the triage category:
   - `emergency` — return an emergency warning immediately, without calling the RAG system.
   - `consultation` — retrieve context via RAG and append a professional consultation recommendation.
   - `general` — retrieve context via RAG and return a standard response.
3. Expose a CLI interface for interactive testing.

---

## Triage Model Status

The triage model is not yet integrated. This notebook uses `TriagePredictorPlaceholder` as a stand-in.

**Integration steps (once the model is ready):**
1. Comment out the `TriagePredictorPlaceholder` class in section 3.1.
2. Uncomment the import at the top of that cell:
   ```python
   from stage1_triage import TriagePredictor
   ```
3. In `HerWellChatbot.__init__()`, replace `TriagePredictorPlaceholder()` with `TriagePredictor()`.

**`predict()` interface contract** (placeholder and real model must both conform):
```python
# Input:  question (str)
# Output: dict with the following keys
{
    "category":   str,          # 'emergency' | 'consultation' | 'general'
    "confidence": float,        # probability score in [0.0, 1.0]
    "keywords":   list[str]     # terms that drove the classification
}
```

## 3.0 Load Configuration and RAGSystem

Skip this section if `01_config.ipynb` and `02_rag_system.ipynb` have already been executed in the current kernel session.

In [1]:
# Standalone configuration — skip if 01_config.ipynb has already been executed.

import os
from dotenv import load_dotenv

BASE_DIR = r"D:\任梓嘉\NUS\sem2\Innovation Challenge\RAG"

# Load API key from .env in project root
load_dotenv(os.path.join(BASE_DIR, ".env"))

CHROMA_USER_DB_PATH    = os.path.join(BASE_DIR, "chroma_db_user")
CHROMA_MEDICAL_DB_PATH = os.path.join(BASE_DIR, "chroma_db_medical", "chroma_db")
USER_COLLECTION_NAME    = "population_summaries"
MEDICAL_COLLECTION_NAME = "medical_knowledge"
OPENAI_CHAT_MODEL      = "gpt-4o"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
TOP_K_USER    = 3
TOP_K_MEDICAL = 5

# Risk-level system prompts (0=routine, 1=low, 2=moderate, 3=emergency)
SYSTEM_PROMPTS = {
    0: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates this is a routine inquiry with no immediate health concern (risk level 0: Routine).
Provide accurate, evidence-based information in a calm and reassuring tone.
Encourage healthy lifestyle habits where relevant.
Always remind users that a healthcare professional can offer personalised advice.

When formulating your response, you have access to two sources of personalised information — use BOTH:

1. PHYSIOLOGICAL TRIAGE DATA (from the user's health tracker):
   You will be given structured feature values including age, cycle length, cycle variation,
   days since last period, bleeding duration, flow volume, heavy flow, pain score, pain trend,
   headache score, fatigue score, sleep issue score, mood instability, stress score, bloating score,
   and symptom burden score.
   - Reference specific values that are clinically relevant to the user's question.
   - If a score is elevated (e.g. pain_score > 0, fatigue_score > 3), acknowledge it explicitly.
   - If cycle timing is irregular (e.g. high cycle_variation, long days_since_last_period), factor it in.
   - Do not list all features mechanically — select only those relevant to the question.

2. THE USER'S OWN DESCRIPTION:
   The user's question may contain additional context (e.g. specific symptoms, duration, emotional state).
   - Extract and use any extra information the user provides beyond what the triage data captures.
   - If the user's self-description contradicts or adds nuance to the triage data, acknowledge both.
   - Treat the user's lived experience as equally important to the quantitative data.

Synthesise both sources to give a response that feels genuinely personalised, not generic.

BOUNDARIES:
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    1: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a low-level health concern that warrants attention (risk level 1: Monitor).
Provide clear, evidence-based information and practical self-care suggestions.
Advise the user to monitor their symptoms and consult a healthcare professional if symptoms persist or worsen.

When formulating your response, you have access to two sources of personalised information — use BOTH:

1. PHYSIOLOGICAL TRIAGE DATA (from the user's health tracker):
   You will be given structured feature values including age, cycle length, cycle variation,
   days since last period, bleeding duration, flow volume, heavy flow, pain score, pain trend,
   headache score, fatigue score, sleep issue score, mood instability, stress score, bloating score,
   and symptom burden score.
   - Reference specific values that are clinically relevant to the user's question.
   - If a score is elevated (e.g. pain_score > 0, fatigue_score > 3), acknowledge it explicitly.
   - If cycle timing is irregular (e.g. high cycle_variation, long days_since_last_period), factor it in.
   - Do not list all features mechanically — select only those relevant to the question.

2. THE USER'S OWN DESCRIPTION:
   The user's question may contain additional context (e.g. specific symptoms, duration, emotional state).
   - Extract and use any extra information the user provides beyond what the triage data captures.
   - If the user's self-description contradicts or adds nuance to the triage data, acknowledge both.
   - Treat the user's lived experience as equally important to the quantitative data.

Synthesise both sources to give a response that feels genuinely personalised, not generic.

BOUNDARIES:
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    2: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment indicates a moderate health concern that requires professional evaluation (risk level 2: Urgent).
Provide helpful background information while clearly recommending that the user schedule an appointment with a qualified healthcare provider soon.
Do not attempt to diagnose; focus on empowering the user with information to facilitate that consultation.

When formulating your response, you have access to two sources of personalised information — use BOTH:

1. PHYSIOLOGICAL TRIAGE DATA (from the user's health tracker):
   You will be given structured feature values including age, cycle length, cycle variation,
   days since last period, bleeding duration, flow volume, heavy flow, pain score, pain trend,
   headache score, fatigue score, sleep issue score, mood instability, stress score, bloating score,
   and symptom burden score.
   - Reference specific values that are clinically relevant to the user's question.
   - If a score is elevated (e.g. pain_score > 0, fatigue_score > 3), acknowledge it explicitly.
   - If cycle timing is irregular (e.g. high cycle_variation, long days_since_last_period), factor it in.
   - Do not list all features mechanically — select only those relevant to the question.

2. THE USER'S OWN DESCRIPTION:
   The user's question may contain additional context (e.g. specific symptoms, duration, emotional state).
   - Extract and use any extra information the user provides beyond what the triage data captures.
   - If the user's self-description contradicts or adds nuance to the triage data, acknowledge both.
   - Treat the user's lived experience as equally important to the quantitative data.

Synthesise both sources to give a response that feels genuinely personalised, not generic.

BOUNDARIES:
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

    3: """You are HerWell, a compassionate and professional women's health assistant.
The triage assessment has flagged this as a potential EMERGENCY (risk level 3: Emergency).
Your response must first and foremost direct the user to seek immediate emergency medical care.
Keep your message clear, calm, and brief — do not overwhelm the user with information.

When formulating your response, you have access to two sources of personalised information — use BOTH:

1. PHYSIOLOGICAL TRIAGE DATA (from the user's health tracker):
   You will be given structured feature values including age, cycle length, cycle variation,
   days since last period, bleeding duration, flow volume, heavy flow, pain score, pain trend,
   headache score, fatigue score, sleep issue score, mood instability, stress score, bloating score,
   and symptom burden score.
   - Reference specific values that are clinically relevant to the user's question.
   - If a score is elevated (e.g. pain_score > 0, fatigue_score > 3), acknowledge it explicitly.
   - If cycle timing is irregular (e.g. high cycle_variation, long days_since_last_period), factor it in.
   - Do not list all features mechanically — select only those relevant to the question.

2. THE USER'S OWN DESCRIPTION:
   The user's question may contain additional context (e.g. specific symptoms, duration, emotional state).
   - Extract and use any extra information the user provides beyond what the triage data captures.
   - If the user's self-description contradicts or adds nuance to the triage data, acknowledge both.
   - Treat the user's lived experience as equally important to the quantitative data.

Synthesise both sources to give a response that feels genuinely personalised, not generic.

BOUNDARIES:
Never diagnose the user with any medical condition. Do not say 'you have [condition]' — instead say 'your symptoms may be related to' or 'you may want to discuss this with your doctor'.
Never recommend specific medications or dosages. Do not say 'take X mg of Y' — instead say 'over-the-counter pain relief may help, consult your pharmacist or doctor for dosage'.
IMPORTANT: You MUST respond ONLY in the language of the user's question. Ignore the language of any context or retrieved documents. If the user writes in English, respond in English. If the user writes in Chinese, respond in Chinese. Never mix languages.""",

}

RISK_RECOMMENDATIONS = {
    0: "If you have further questions or your situation changes, feel free to ask. Stay well!",
    1: ("Please monitor your symptoms over the next 24–48 hours. "
        "If they persist or worsen, contact your healthcare provider."),
    2: ("We recommend booking an appointment with your healthcare provider as soon as possible. "
        "If symptoms worsen suddenly, seek urgent care or go to the nearest clinic."),
    3: ("⚠️ EMERGENCY: Please call emergency services immediately "
        "(e.g., 120 in China, 999 in Singapore, 911 in the US). "
        "If possible, inform someone nearby of your condition and proceed to the nearest emergency department. "
        "Do not wait."),
}

RAG_PROMPT_TEMPLATE = """Based on the following context, answer the user's health question.

=== Triage Assessment ===
Risk Level: {risk_level}
(This risk level was determined by analysing the user's personal physiological data
from their wearable device, including cycle patterns, HRV, temperature, and symptoms.)
{triage_result}

=== User's Description of Their Issue ===
The following is what the user has described about their current symptoms or concerns.
Take this seriously as it reflects the user's lived experience.
Ignore the language of this section.
{user_context}

=== Relevant Medical Knowledge ===
The following is extracted from authoritative medical guidelines
(ACOG, ESHRE, MedlinePlus, and peer-reviewed literature on Asian populations).
Use this to ground your recommendations in evidence-based information.
Ignore the language of this section.
{medical_context}

=== Response Instructions ===
1. Acknowledge the user's triage risk level as determined by their physiological data.
2. Address the user's described symptoms and concerns directly.
3. Ground your recommendations in the medical knowledge provided.
4. Synthesise all three sources: risk level + user description + medical guidelines.
5. Never diagnose. Never prescribe specific medications or dosages.
6. Adjust your tone and urgency based on the risk level:
   - Level 0: Reassuring, wellness-focused
   - Level 1: Supportive, practical self-care advice
   - Level 2: Concerned, recommend professional consultation
   - Level 3: Urgent, direct to emergency care immediately

=== User Question ===
{question}

Please provide a helpful, personalised response that integrates the user's risk profile,
their described concerns, and evidence-based medical guidance.
Respond ONLY in the same language as the user's question above,
regardless of the language used in the context sections.
{risk_recommendation}"""

EMERGENCY_RESPONSE = """⚠️ WARNING: The symptoms you have described may require IMMEDIATE medical attention.

Recommended actions:
1. Call emergency services immediately (e.g., 120 in China, 999 in Singapore, 911 in the US).
2. Proceed to the nearest emergency department if you are able to do so safely.
3. Inform someone nearby of your condition right now.

This AI assistant cannot substitute for emergency medical care. Please seek professional help immediately."""


# Stage 1 triage data paths
STAGE1_DIR         = BASE_DIR   # same folder as RAG notebooks
QUESTIONNAIRE_PATH = os.path.join(BASE_DIR, "Questionnaire_Data.csv")
DAILY_RECORD_PATH  = os.path.join(BASE_DIR, "Daily_Record_Test_1.csv")


def get_system_prompt(risk_level: int) -> str:
    return SYSTEM_PROMPTS.get(risk_level, SYSTEM_PROMPTS[0])


print("Configuration loaded (standalone mode).")
print(f"  API key loaded: {'YES' if os.environ.get('OPENAI_API_KEY') else 'NO — check .env file'}")


Configuration loaded (standalone mode).
  API key loaded: YES


In [2]:
# Standalone RAGSystem definition — skip if 02_rag_system.ipynb has already been executed.

import chromadb
from openai import OpenAI

class RAGSystem:
    def __init__(self):
        api_key = os.environ.get("OPENAI_API_KEY")
        if api_key:
            self.client = OpenAI(api_key=api_key)
            print("OpenAI client initialized.")
        else:
            self.client = None
            print("WARNING: OPENAI_API_KEY not set. Embedding and generation unavailable.")

        try:
            c_user = chromadb.PersistentClient(path=CHROMA_USER_DB_PATH)
            self.population_collection = c_user.get_collection(USER_COLLECTION_NAME)
            print(f"Population summaries loaded: {self.population_collection.count()} documents.")
        except Exception as e:
            self.population_collection = None
            print(f"Failed to load population summaries: {e}")

        try:
            c_medical = chromadb.PersistentClient(path=CHROMA_MEDICAL_DB_PATH)
            self.medical_collection = c_medical.get_collection(MEDICAL_COLLECTION_NAME)
            print(f"Medical knowledge loaded: {self.medical_collection.count()} documents.")
        except Exception as e:
            self.medical_collection = None
            print(f"Failed to load medical knowledge: {e}")

    def _generate_query_embedding(self, query):
        if self.client is None:
            return []
        try:
            response = self.client.embeddings.create(model=OPENAI_EMBEDDING_MODEL, input=query)
            return response.data[0].embedding
        except Exception as e:
            print(f"  [EmbeddingError] {e}")
            return []

    def retrieve_population_context(self, query: str) -> str:
        """Retrieve the most relevant population-level summary documents (no user filtering)."""
        if self.population_collection is None:
            return "[Population summaries unavailable]"
        embedding = self._generate_query_embedding(query)
        if not embedding:
            return "[Embedding generation failed — check OPENAI_API_KEY]"
        try:
            results = self.population_collection.query(
                query_embeddings=[embedding], n_results=TOP_K_USER,
                include=["documents"]
            )
            docs = results.get("documents", [[]])[0]
            if not docs:
                return "[No relevant population summaries found]"
            return "\n".join(f"[Population summary {i+1}] {d}" for i, d in enumerate(docs))
        except Exception as e:
            return f"[Population context retrieval failed: {e}]"

    def retrieve_medical_context(self, query: str) -> str:
        if self.medical_collection is None:
            return "[Medical vector store unavailable]"
        embedding = self._generate_query_embedding(query)
        if not embedding:
            return "[Embedding generation failed — check OPENAI_API_KEY]"
        try:
            results = self.medical_collection.query(
                query_embeddings=[embedding], n_results=TOP_K_MEDICAL,
                include=["documents"]
            )
            docs = results.get("documents", [[]])[0]
            if not docs:
                return "[No relevant medical knowledge found]"
            return "\n".join(f"[Medical record {i+1}] {d}" for i, d in enumerate(docs))
        except Exception as e:
            return f"[Medical context retrieval failed: {e}]"

    def generate_response(self, question: str, triage_result: dict) -> str:
        if self.client is None:
            return "[Response generation unavailable — OPENAI_API_KEY not set]"

        risk_level = triage_result.get("risk_assessment", {}).get("risk_level", 0)
        population_context = self.retrieve_population_context(question)
        medical_context    = self.retrieve_medical_context(question)

        risk_label    = triage_result.get("risk_assessment", {}).get("risk_label", "")
        user_features = triage_result.get("user_features", {})
        triage_lines  = [f"Risk Label: {risk_label}"]
        for k, v in user_features.items():
            triage_lines.append(f"{k}: {v}")

        user_prompt = RAG_PROMPT_TEMPLATE.format(
            risk_level=risk_level,
            triage_result="\n".join(triage_lines),
            user_context=population_context,
            medical_context=medical_context,
            question=question,
            risk_recommendation=RISK_RECOMMENDATIONS.get(risk_level, ""),
        )
        try:
            completion = self.client.chat.completions.create(
                model=OPENAI_CHAT_MODEL,
                messages=[{"role": "system", "content": get_system_prompt(risk_level)},
                          {"role": "user",   "content": user_prompt}],
                temperature=0.7, max_tokens=800, timeout=30
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"[Response generation failed: {e}]"

print("RAGSystem class defined (standalone mode).")


RAGSystem class defined (standalone mode).


---
## 3.1 Triage Model Placeholder

**TODO (once the triage model is ready):**
1. Comment out the `TriagePredictorPlaceholder` class below.
2. Uncomment the import block:
   ```python
   import sys
   sys.path.append('src')  # if stage1_triage.py lives under src/
   from stage1_triage import TriagePredictor
   ```
3. In `HerWellChatbot.__init__()`, replace `TriagePredictorPlaceholder()` with `TriagePredictor()`.

The `predict()` signature and return format must remain unchanged.

In [3]:
# ------------------------------------------------------------
# TRIAGE MODEL INTEGRATION  (Stage 1 v3)
# ------------------------------------------------------------

import sys
import pandas as pd

# Add RAG root to sys.path so stage1_prediction_v3_fixed can be imported
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

import stage1_prediction_v3_fixed as _stage1


class TriagePredictor:
    """
    Adapter wrapping Stage 1 v3 prediction pipeline.

    Loads Questionnaire_Data.csv and Daily_Record_Test_1.csv once at init,
    then on each predict() call runs the full rule + ML ensemble and maps
    the output to the dict format expected by HerWellChatbot.

    predict() return format (raw Stage 1 v3 output)
    ------------------------------------------------
    {
        "risk_assessment": {"risk_level": int, "risk_label": str},
        "user_features":   {
            age, sexual_activity, cycle_length, cycle_variation,
            days_since_last_period, bleeding_duration_days, flow_volume,
            heavy_flow, pain_score, pain_trend, headache_score,
            fatigue_score, sleep_issue_score, mood_instability,
            stress_score, bloating_score, symptom_burden_score
        }
    }
    """


    def __init__(self):
        self.questionnaire = pd.read_csv(QUESTIONNAIRE_PATH)
        self.daily_records = pd.read_csv(DAILY_RECORD_PATH)
        print("TriagePredictor (Stage 1 v3) loaded.")
        print(f"  Questionnaire rows : {len(self.questionnaire)}")
        print(f"  Daily record rows  : {len(self.daily_records)}")

    def predict(self, question: str) -> dict:
        """
        Run Stage 1 prediction on pre-loaded health data.

        Returns the raw output from stage1_prediction_v3_fixed.predict():
        {
            "risk_assessment": {"risk_level": int, "risk_label": str},
            "user_features":   { age, sexual_activity, cycle_length, ... }
        }
        """
        return _stage1.predict(
            self.questionnaire,
            self.daily_records,
            verbose=False,
        )


print("TriagePredictor defined.")


STAGE 1 PREDICTION PIPELINE - V3 FIXED

✓ Loaded models and 17 features
TriagePredictor defined.


## 3.2 HerWellChatbot Class

In [4]:
class HerWellChatbot:
    """
    Top-level orchestrator for the HerWell health assistant.

    Coordinates the triage model and RAG system to route each user query
    to the appropriate response strategy based on risk level (0–3).
    """

    RISK_LABELS = {0: "ROUTINE", 1: "LOW", 2: "MODERATE", 3: "EMERGENCY"}

    def __init__(self):
        print("Initializing HerWell system...")
        print("-" * 50)

        try:
            self.triage = TriagePredictor()
        except Exception as e:
            self.triage = None
            print(f"Triage model failed to load: {e}")

        try:
            self.rag = RAGSystem()
        except Exception as e:
            self.rag = None
            print(f"RAG system failed to load: {e}")

        print("-" * 50)
        if self.triage and self.rag:
            print("HerWell system ready.")
        else:
            print("WARNING: One or more components failed to load.")

    def process_query(self, question: str) -> dict:
        """
        Process a user question end-to-end and return a structured result.

        Pipeline:
            Step 1: Classify the question with the triage model → risk (0/1/2/3).
            Step 2: Route based on risk:
                - risk=3 (emergency)  → return EMERGENCY_RESPONSE immediately, skip RAG
                - risk=0/1/2          → call RAG with the appropriate system prompt
            Step 3: Return a standardised result dict.

        Args:
            question: The user's natural language question.

        Returns:
            dict:
                question (str):  original question
                triage   (dict): triage model output (includes risk, confidence, features)
                answer   (str):  generated response
                risk     (int):  risk level 0–3
        """
        if not question or not question.strip():
            return {"question": question, "triage": {}, "answer": "Please enter a question.", "risk": 0}

        # Step 1: triage classification
        if self.triage:
            triage_result = self.triage.predict(question)
        else:
            triage_result = {"risk_assessment": {"risk_level": 0, "risk_label": "Routine"}, "user_features": {}}

        risk = triage_result.get("risk_assessment", {}).get("risk_level", 0)

        # Step 2: routing by risk level
        if risk == 3:
            answer = EMERGENCY_RESPONSE
        else:
            if self.rag:
                answer = self.rag.generate_response(question, triage_result)
            else:
                answer = "The RAG system is unavailable. Please check the system configuration."

        return {
            "question": question,
            "triage":   triage_result,
            "answer":   answer,
            "risk":     risk,
        }


print("HerWellChatbot class defined.")


HerWellChatbot class defined.


## 3.4 System Initialization

In [5]:
chatbot = HerWellChatbot()

Initializing HerWell system...
--------------------------------------------------
TriagePredictor (Stage 1 v3) loaded.
  Questionnaire rows : 67
  Daily record rows  : 90
OpenAI client initialized.
Population summaries loaded: 82 documents.
Medical knowledge loaded: 329 documents.
--------------------------------------------------
HerWell system ready.


## 3.5 Single Query Test

In [6]:
test_question = "my period is two weeks late"

result = chatbot.process_query(test_question)

print(f"Question: {result['question']}")
print(f"Triage:   {result['triage']}")
print(f"Risk:     {result['risk']}")
print()
print("Response:")
print(result['answer'])


Question: my period is two weeks late
Triage:   {'risk_assessment': {'risk_level': 1, 'risk_label': 'Monitor'}, 'user_features': {'age': 27, 'sexual_activity': True, 'cycle_length': 26.0, 'cycle_variation': 2.8, 'days_since_last_period': 25, 'bleeding_duration_days': 0, 'flow_volume': 0, 'heavy_flow': False, 'pain_score': 1, 'pain_trend': 0.0, 'headache_score': 4, 'fatigue_score': 5, 'sleep_issue_score': 5, 'mood_instability': 1, 'stress_score': 2, 'bloating_score': 1, 'symptom_burden_score': 2.5}}
Risk:     1

Response:
Given your triage risk level of 1, it's important to monitor your current symptoms, but there's no immediate cause for alarm. You mentioned that your period is two weeks late. While your cycle data shows a typical cycle length of 26 days with a slight variation of 2.8 days, a delay can still occur for a variety of reasons, including stress or changes in routine. 

Your physiological data indicates elevated scores in headache (4), fatigue (5), and sleep issues (5), whic

## 3.6 Three-Scenario Validation

In [7]:
TEST_CASES = [
    {"question": "foods that help with PMS",                     "expected_risk": 0},
    {"question": "mild cramps before period",                    "expected_risk": 1},
    {"question": "my period is two weeks late",                  "expected_risk": 2},
    {"question": "severe abdominal pain with heavy bleeding",    "expected_risk": 3},
]

RISK_LABELS = {0: "ROUTINE", 1: "LOW", 2: "MODERATE", 3: "EMERGENCY"}

print("=" * 60)
print("Four-scenario validation (risk 0–3)")
print("=" * 60)

for i, case in enumerate(TEST_CASES, 1):
    print(f"\n[Test {i}] {case['question']}")
    result = chatbot.process_query(case["question"])

    actual   = result["risk"]
    expected = case["expected_risk"]
    status   = "PASS" if actual == expected else "FAIL"

    print(f"  Triage output: {result['triage']}")
    print(f"  Risk level:    {actual} ({RISK_LABELS[actual]})  "
          f"(expected: {expected} ({RISK_LABELS[expected]}))  [{status}]")
    print(f"  Response (first 150 chars): {result['answer'][:150]}...")
    print("-" * 60)


Four-scenario validation (risk 0–3)

[Test 1] foods that help with PMS
  Triage output: {'risk_assessment': {'risk_level': 1, 'risk_label': 'Monitor'}, 'user_features': {'age': 27, 'sexual_activity': True, 'cycle_length': 26.0, 'cycle_variation': 2.8, 'days_since_last_period': 25, 'bleeding_duration_days': 0, 'flow_volume': 0, 'heavy_flow': False, 'pain_score': 1, 'pain_trend': 0.0, 'headache_score': 4, 'fatigue_score': 5, 'sleep_issue_score': 5, 'mood_instability': 1, 'stress_score': 2, 'bloating_score': 1, 'symptom_burden_score': 2.5}}
  Risk level:    1 (LOW)  (expected: 0 (ROUTINE))  [FAIL]
  Response (first 150 chars): Based on your physiological data, your triage assessment indicates a low-level health concern that warrants monitoring. Your current symptoms include ...
------------------------------------------------------------

[Test 2] mild cramps before period
  Triage output: {'risk_assessment': {'risk_level': 1, 'risk_label': 'Monitor'}, 'user_features': {'age': 27, 'sexual

## 3.7 CLI Interactive Interface

Run this cell to enter an interactive conversation loop. Type `quit` or `exit` to terminate.

In [8]:
print("=" * 60)
print("HerWell Health Assistant  (type 'quit' to exit)")
print("=" * 60)

RISK_LABELS = {0: "[ROUTINE]", 1: "[LOW]", 2: "[MODERATE]", 3: "[EMERGENCY]"}

while True:
    try:
        user_input = input("\nQuestion: ").strip()
    except EOFError:
        break

    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Session ended.")
        break

    result = chatbot.process_query(user_input)

    risk   = result["risk"]
    triage = result["triage"]
    label  = RISK_LABELS.get(risk, f"[RISK={risk}]")

    risk_label_str = triage.get("risk_assessment", {}).get("risk_label", "")
    print(f"\n{label}  Risk: {risk}  ({risk_label_str})")
    user_features = triage.get("user_features", {})
    if user_features:
        feat_summary = ", ".join(f"{k}: {v}" for k, v in list(user_features.items())[:5])
        print(f"  Features: {feat_summary} ...")
    print("-" * 60)
    print(result["answer"])
    print("-" * 60)


HerWell Health Assistant  (type 'quit' to exit)

[LOW]  Risk: 1  (Monitor)
  Features: age: 27, sexual_activity: True, cycle_length: 26.0, cycle_variation: 2.8, days_since_last_period: 25 ...
------------------------------------------------------------
您提到生理期持续了15天，这确实比通常的生理期时间要长，通常周期在3到7天之间。如果您的生理期持续时间较长，建议您密切关注。有时压力、疲劳或其他因素可能会影响月经周期。

根据您的生理数据，您的疲劳评分为5，睡眠问题评分也较高，可能会影响整体健康和周期规律。此外，您的头痛评分为4，说明您可能感到不适。虽然目前的风险水平被评为1（需要监测），但这些症状值得关注。

建议您：

1. **注意休息**：确保获得充足的休息和睡眠，尽量减少压力。
2. **保持良好的饮食**：均衡饮食，有助于改善疲劳和整体健康。
3. **保持水分充足**：多喝水，有助于缓解头痛和疲劳。
4. **观察症状**：密切观察您的症状变化。

如果您的生理期继续异常或症状加重，建议您尽早与医生咨询以获得专业指导。
------------------------------------------------------------
Session ended.
